[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/07_low_precision.ipynb)

# 7 · Low-precision matmuls: FP16, BF16, FP8 and the cost of each
Weeks 6, 18 and 23. Measure speed and error of matmuls in different formats. **FP8 needs an Ada/Hopper GPU** (Colab L4 or better;
T4 will skip that part). FP8 uses per-tensor scaling here; DeepSeek-V3's recipe uses finer 1×128 / 128×128 scaling (week 19).

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
def timed(f, n=50):
    for _ in range(5): f()
    torch.cuda.synchronize(); s, e = torch.cuda.Event(True), torch.cuda.Event(True); s.record()
    for _ in range(n): f()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e) / n
N = 4096
a32, b32 = torch.randn(N, N, device="cuda"), torch.randn(N, N, device="cuda")
ref = a32.double() @ b32.double()
def rel(c): return ((c.double() - ref).norm() / ref.norm()).item()
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    a, b = a32.to(dt), b32.to(dt)
    t = timed(lambda: a @ b)
    print(f"{str(dt):>15}: {2 * N**3 / t / 1e9:7.1f} TFLOP/s   rel. error {rel(a @ b):.2e}")

In [ ]:
cap = torch.cuda.get_device_capability()
if cap >= (8, 9) and hasattr(torch, "_scaled_mm"):
    f8 = torch.float8_e4m3fn
    sa, sb = a32.abs().max() / 448, b32.abs().max() / 448
    a8 = (a32 / sa).to(f8)
    b8 = (b32 / sb).to(f8).t().contiguous().t()          # _scaled_mm wants column-major B
    one = lambda x: torch.tensor(x, device="cuda", dtype=torch.float32)
    mm = lambda: torch._scaled_mm(a8, b8, scale_a=one(sa.item()), scale_b=one(sb.item()), out_dtype=torch.bfloat16)
    t = timed(mm)
    print(f"FP8 e4m3 (per-tensor scale): {2 * N**3 / t / 1e9:.1f} TFLOP/s   rel. error {rel(mm()):.2e}")
else:
    print(f"GPU capability {cap}: no FP8 Tensor Cores here (needs ≥ 8.9, e.g. L4/H100) — skipped")

## Your turn
1. Add a few outlier columns to `a32` (×100) and compare FP8 per-tensor error with the 1×128 tile scaling from your week-19
   exercise (quantise/dequantise per tile, then multiply in BF16 to simulate).
2. Why does BF16 have larger error than FP16 here, yet is preferred for training?
3. Estimate the memory and bandwidth savings of FP8 KV caches for a 32-layer, 8-KV-head model at 32K context.